# pfpoly - Google Colab Setup

TDBB (Time-Dependent Bond Boosting) ベースの重合シミュレーション環境をColab上に構築します。

**対応バックエンド:**
- OrbMol-v2 (orb-models)
- MACE-MP-0

**GPU:** Colab T4 (無料) / A100 (Pro) で動作確認済み

## 0. GPU確認

**ランタイム → ランタイムのタイプを変更 → GPU** を選択してください。

In [ ]:
!nvidia-smi

## 1. リポジトリのクローンとインストール

In [ ]:
import os

REPO_URL = 'https://github.com/M-Umeda316/pfpoly.git'  # TODO: 実際のリポジトリURLに変更
REPO_DIR = '/content/pfpoly'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

## 2. 依存パッケージのインストール

PyTorch (CUDA 12.x) → pfpoly + バックエンド の順にインストールします。  
全体で5〜10分程度かかります。

In [ ]:
# PyTorch (Colab標準のCUDA 12.xに合わせる)
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128

In [ ]:
# pfpoly本体 + 全バックエンド
!pip install -q -e '.[mace,orb,plot,fit,rdkit]'

In [ ]:
# インストール確認
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

import ase
print(f'ASE: {ase.__version__}')

import numpy as np
print(f'NumPy: {np.__version__}')

## 3. バックエンド動作確認

In [ ]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

In [ ]:
# MACE-MP-0 テスト
try:
    from src.backends.mace_backend import create_mace_calculator
    calc = create_mace_calculator(model='small', device='cuda')
    print(f'MACE backend OK: {calc.name}')
except Exception as e:
    print(f'MACE backend error: {e}')

In [ ]:
# OrbMol-v2 テスト
try:
    from src.backends.orb_backend import create_orb_calculator
    calc = create_orb_calculator(device='cuda')
    print(f'ORB backend OK: {calc.name}')
except Exception as e:
    print(f'ORB backend error: {e}')

## 4. Google Driveマウント（結果の永続化）

セッション切断後も結果を保持するため、Google Driveに出力します。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_OUTPUT = '/content/drive/MyDrive/pfpoly_runs'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
print(f'Output directory: {DRIVE_OUTPUT}')

## 5. スモークテスト（小規模実行）

3モノマー + 1開始剤、3サイクルの最小構成で動作確認します。  
GPU使用で1〜2分程度。

In [ ]:
!python scripts/run_vinyl_aibn.py \
    --seed 42 \
    --n-monomers 3 \
    --n-initiators 1 \
    --n-cycles 3 \
    --biased-steps 200 \
    --unbiased-steps 200 \
    --backend mace \
    --device cuda \
    --output-dir runs/smoke_colab

In [ ]:
# 結果確認
import json

summary_path = 'runs/smoke_colab/summary.json'
if os.path.exists(summary_path):
    with open(summary_path) as f:
        summary = json.load(f)
    print(f'Total steps: {summary["total_steps"]}')
    print(f'Atoms: {summary["n_atoms"]}')
    print(f'Box: {summary["box_size_A"]:.2f} Å')
    print(f'Formations: {summary["confirmed_formations"]}')
    print(f'Dissociations: {summary["confirmed_dissociations"]}')
else:
    print('Summary not found — check the run output above for errors.')

## 6. 論文スケール実行（オプション）

論文準拠のパラメータで実行します。出力はGoogle Driveへ。  
**注意:** 無料Colabの場合、セッション切断に注意してください（12時間上限）。

In [ ]:
# 論文 Table S1 準拠パラメータ (vinyl polymerization)
# 必要に応じて調整してください

SEED = 7
N_MONOMERS = 20
N_INITIATORS = 4
N_CYCLES = 30
BIASED_STEPS = 2000
UNBIASED_STEPS = 2000
BACKEND = 'orb'       # 'orb' or 'mace'
TEMPERATURE = 333.0   # K

OUTPUT_DIR = f'{DRIVE_OUTPUT}/vinyl_{BACKEND}_seed{SEED}'

!python scripts/run_vinyl_aibn.py \
    --seed {SEED} \
    --n-monomers {N_MONOMERS} \
    --n-initiators {N_INITIATORS} \
    --n-cycles {N_CYCLES} \
    --biased-steps {BIASED_STEPS} \
    --unbiased-steps {UNBIASED_STEPS} \
    --backend {BACKEND} \
    --device cuda \
    --temperature {TEMPERATURE} \
    --density 0.5 \
    --output-dir {OUTPUT_DIR}

## 7. 図の生成

In [ ]:
# スモークテスト結果の可視化
RUN_DIR = 'runs/smoke_colab'  # 論文スケールの場合は OUTPUT_DIR に変更

!python scripts/reproduce_figures.py \
    --trajectory {RUN_DIR}/trajectory.jsonl \
    --bonds {RUN_DIR}/bonds.jsonl \
    --n-reactive-sites 3 \
    --target-temperature 333 \
    --output-dir {RUN_DIR}/figures

In [ ]:
# 図の表示
from IPython.display import Image, display
from pathlib import Path

fig_dir = Path(RUN_DIR) / 'figures'
if fig_dir.exists():
    for fig in sorted(fig_dir.glob('*.png')):
        print(f'\n--- {fig.name} ---')
        display(Image(filename=str(fig), width=600))
else:
    print('No figures found.')

## 8. 結果をDriveにコピー

In [ ]:
import shutil

LOCAL_RUN = 'runs/smoke_colab'
DRIVE_DEST = f'{DRIVE_OUTPUT}/smoke_colab'

if os.path.exists(LOCAL_RUN):
    shutil.copytree(LOCAL_RUN, DRIVE_DEST, dirs_exist_ok=True)
    print(f'Copied to {DRIVE_DEST}')
else:
    print('No local run to copy.')

## テスト実行（オプション）

In [ ]:
!python -m pytest tests/ -x -q --tb=short -m 'not slow'